In [ ]:
import os
import sys
import json
import copy
import base64
import re
from datetime import datetime

from openai import OpenAI
import yaml
import numpy as np

sys.path.append("../")
from src.utils.parser import *
from src.utils.feature_utils import theta_to_language, theta_to_state_mask

# Set your OpenAI API key
try:
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
except KeyError:
    # read from ~/.openai_api_key
    with open(os.path.expanduser("~/.openai_api_key"), "r") as f:
        api_key = f.read().strip()
    client = OpenAI(api_key=api_key)

%load_ext autoreload
%autoreload 2

In [ ]:
task_instruction = "The task is to move a coffee grasped with the robot's end effector where there is a human user nearby."


def encode_image(image_path: str) -> str:
    """Encode a local image file as a base64 data URL for the OpenAI vision API."""
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    ext = os.path.splitext(image_path)[1].lstrip(".").lower() or "png"
    if ext == "jpg":
        ext = "jpeg"
    return f"data:image/{ext};base64,{encoded}"


def build_image_content(traj_image_paths):
    """Turn a list of image paths or URLs into OpenAI chat image content entries."""
    content = []
    for path in traj_image_paths:
        if path.startswith("http://") or path.startswith("https://") or path.startswith("data:"):
            url = path
        else:
            url = encode_image(path)
        content.append({"type": "image_url", "image_url": {"url": url}})
    return content


def is_instruction_ambiguous(
    instruction: str,
    model_text: str = "gpt-4o-mini",
) -> dict:
    """Check whether an instruction is ambiguous.
    
    Returns a dict: {"ambiguous": bool, "explanation": str}.
    """
    messages = [
        {"role": "system", "content": (
            "You are interfacing with a robotics environment in which a robotic arm is sitting on a table. "
            "There is also a laptop on the table and a human standing next to the table. The robotic arm is "
            "learning to manipulate a cup based on some language command (e.g. 'Stay close to the table "
            "surface. Carry the cup upright.'). The user will specify the command that they would like to "
            "teach the robot. However, sometimes the user command is ambiguous, for example it may be "
            "missing the referent (e.g., 'Stay close'), or the desired behavior (e.g., 'Cup'). You will be "
            "asked to assess if the user command is ambiguous or not. Please respond ONLY with JSON: "
            '{"ambiguous":<true|false>,"explanation":...}'
        )},
        {"role": "user", "content": f"Instruction:\n{instruction}"},
    ]
    resp = client.chat.completions.create(
        model=model_text,
        messages=messages,
        temperature=0,
    )
    content = resp.choices[0].message.content.strip()
    if content.startswith("```json"):
        content = content[len("```json"):]
    if content.startswith("```"):
        content = content[3:]
    if content.endswith("```"):
        content = content[:-3]
    return json.loads(content.strip())


def disambiguate_instruction_vlm(
    instruction: str,
    traj_image_paths: list,
    model_vision: str = "gpt-4o",
    temperature: float = 0.3,
) -> list:
    """Rewrite an ambiguous instruction into clear directive(s) using trajectory images.
    
    Returns a JSON list of 1-2 candidate disambiguated instructions.
    """
    system_content = (
        "You are a vision-language assistant interfacing with a robotics environment.\n\n"
        "Environment:\n"
        "A robotic arm is sitting on a table. There is also a laptop on the table and a human standing next "
        "to the table. The robotic arm is learning to manipulate a cup based on some language command "
        "(e.g. 'Stay close to the table surface. Carry the cup upright.'). The user will specify the command "
        "that they would like to teach the robot. Sometimes the user's command is ambiguous, for example it "
        "may be missing the referent (e.g., 'Stay close'), or the desired behavior (e.g., 'Cup').\n\n"
        "Task:\n"
        "You will be asked to help disambiguate the command using images of the trajectory. Please look at "
        "the trajectory, and do your best to disambiguate what the command means, by either filling in the "
        "referent (e.g., 'Stay close.' -> 'Stay close to the table.' or 'Stay close to the human.') OR by "
        "filling in the desired behavior (e.g., 'Cup' -> 'Keep cup upside down.' or 'Keep cup upright'). "
        "Please ensure your output directly relates to the elements of the environment, such as the human, "
        "the table, or the objects on the table. Be specific about the objects you are referencing, for "
        "example say 'Stay away from the laptop', NOT 'Stay away from the object on the table'. Each "
        "command should only refer to a single object, such as the table or the laptop.\n\n"
        "Output:\n"
        "Walk through your reasoning based on the input command and the images. Prefer a single, "
        "most-supported interpretation; if two interpretations are equally plausible, output exactly two. "
        "On the FINAL LINE, output ONLY a JSON list of 1-2 strings with the plausible disambiguated "
        "command(s). Examples:\n"
        '["Stay close to the table surface"]\n'
        '["Stay close to the human", "Stay close to the laptop"]'
    )
    user_content = [{"type": "text", "text": f"Original Instruction:\n{instruction}"}]
    user_content.extend(build_image_content(traj_image_paths))

    messages = [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
    ]
    resp = client.chat.completions.create(
        model=model_vision,
        messages=messages,
        temperature=temperature,
    )
    content = resp.choices[0].message.content.strip()
    last_line = content.splitlines()[-1].strip()
    return json.loads(last_line)


def process_instruction(
    instruction: str,
    traj_image_paths: list,
    model_text: str = "gpt-4o-mini",
    model_vision: str = "gpt-4o",
    temperature: float = 0.3,
    max_trials: int = 3,
) -> dict:
    """Check ambiguity (text-only), then disambiguate using images if needed."""
    ambiguity = None
    for trial in range(max_trials):
        try:
            ambiguity = is_instruction_ambiguous(instruction, model_text=model_text)
            break
        except Exception as e:
            print(f"Ambiguity check error (trial {trial + 1}/{max_trials}): {e}")
    if ambiguity is None:
        return None

    clear_instruction = [instruction]
    if ambiguity.get("ambiguous"):
        for trial in range(max_trials):
            try:
                clear_instruction = disambiguate_instruction_vlm(
                    instruction,
                    traj_image_paths=traj_image_paths,
                    model_vision=model_vision,
                    temperature=temperature,
                )
                break
            except Exception as e:
                print(f"Disambiguation error (trial {trial + 1}/{max_trials}): {e}")
        else:
            return None

    return {
        "ambiguous": ambiguity.get("ambiguous", False),
        "explanation": ambiguity.get("explanation", ""),
        "disambiguated_instruction": clear_instruction,
    }

In [ ]:
def get_theta_key(theta):
    """Generate an interpretable key for a given theta vector."""
    return '_'.join(str(x) for x in theta)


# Load configuration files
human_config = "../config/humans/frankarobot_multiple_humans.yaml"
with open(human_config, "r") as stream:
    humans_params_list = yaml.safe_load(stream)

config = "../config/reward_learning/obj20_sg10_persg5/maskedrl_llm_mask.yaml"
with open(config, "r") as stream:
    params = yaml.safe_load(stream)

# Load data split configuration
indices_file = os.path.join(params["irl"]["data_split_config_path"], "split_indices.json")
all_trajs, train_trajs, test_trajs = load_split_data(
    params["env"]["trajset_file"],
    params["env"]["per_SG"],
    params["env"]["train_test_split"],
    indices_file=indices_file,
)
print(f"Total trajectories: {len(all_trajs)} | Train: {len(train_trajs)} | Test: {len(test_trajs)}")

seed = 12345
set_seed(seed)

# Load demo indices for each theta
demo_queries = 10
demo_indices_file = os.path.join(params["irl"]["data_split_config_path"], f"demo_indices_100_{seed}.json")
with open(demo_indices_file, "r") as f:
    saved_demo_indices = json.load(f)
    for key in saved_demo_indices.keys():
        saved_demo_indices[key] = saved_demo_indices[key][:demo_queries]
print(f"Loaded demo indices from {demo_indices_file}")

# Configure where trajectory images live. Set `traj_image_folder` to a local directory of rendered
# trajectory views, or leave it as `None` and fill in `traj_image_url_template` with a hosted URL pattern.
# Files are expected to be named as f"traj_{demo_idx}_view_{view}.png".
traj_image_folder = None  # e.g. "../data/traj_viz/<your_viz_folder>"
traj_image_url_template = "<YOUR_IMAGE_URL_TEMPLATE>"  # e.g. "https://example.com/traj_{demo_idx}_view_{view}.png"
camera_viewpoints = [0, 1, 2]  # 0: topdown, 1: diagonal, 2: side


def get_traj_image_paths(demo_idx: int):
    """Return the list of image paths (or URLs) for a given demonstration."""
    if traj_image_folder is not None:
        paths = [
            os.path.join(traj_image_folder, f"traj_{demo_idx}_view_{view}.png")
            for view in camera_viewpoints
        ]
        return [p for p in paths if os.path.exists(p)]
    if "{demo_idx}" not in traj_image_url_template:
        raise ValueError(
            "Set `traj_image_folder` to a local viz directory, or fill in `traj_image_url_template` "
            "with a URL pattern containing {demo_idx} and {view} placeholders."
        )
    return [
        traj_image_url_template.format(demo_idx=demo_idx, view=view)
        for view in camera_viewpoints
    ]

# Disambiguate Ambiguous Language Instructions with a VLM

For each human, take the ambiguous instruction variants (`omit_referent` and `omit_expression`) and use a
vision-language model over rendered trajectory images to produce disambiguated instructions.

In [ ]:
results_list = copy.deepcopy(humans_params_list["humans"])
resume_path = None
previous_resume_path = None
temperature = 0.3
state_dim = 19
model_vision = "gpt-4o"
model_text = "gpt-4o-mini"

# Load resume data if specified
if resume_path is not None:
    with open(resume_path, "r") as f:
        resume_results_list = json.load(f)
    print(f"Resuming from {resume_path}")

ambiguity_types = ["omit_referent", "omit_expression"]

for human_idx, human_params in enumerate(humans_params_list["humans"]):
    # Skip if already processed (when resuming)
    if (
        resume_path is not None
        and "disambiguated_instructions" in resume_results_list[human_idx]
    ):
        results_list[human_idx] = resume_results_list[human_idx]
        continue

    print(f"Processing human {human_idx}/{len(humans_params_list['humans'])}")

    theta = human_params["preferencer"]["theta"]
    theta_key = get_theta_key(theta)
    gt_state_mask = theta_to_state_mask(theta, state_dim=state_dim).astype(int)
    language_instruction = theta_to_language([theta])[0]

    print(f"Human theta: {theta}")
    print(f"Language instruction: {language_instruction}")

    results_list[human_idx]["language_instruction"] = language_instruction
    results_list[human_idx]["theta_key"] = theta_key
    results_list[human_idx]["gt_state_mask"] = gt_state_mask

    if theta_key not in saved_demo_indices:
        print(f"No demo indices for theta {theta_key}, skipping")
        continue

    demo_indices = saved_demo_indices[theta_key]
    results_list[human_idx]["demo_indices"] = demo_indices
    results_list[human_idx]["disambiguated_instructions"] = {
        a: {"ambiguous_instruction": None, "per_demo": {}} for a in ambiguity_types
    }

    for ambiguity_type in ambiguity_types:
        ambiguous_instruction = theta_to_language([theta], language_ambiguity=ambiguity_type)[0]
        print(f"  [{ambiguity_type}] Ambiguous instruction: {ambiguous_instruction}")
        results_list[human_idx]["disambiguated_instructions"][ambiguity_type][
            "ambiguous_instruction"
        ] = ambiguous_instruction

        for demo_idx in demo_indices:
            traj_image_paths = get_traj_image_paths(demo_idx)
            if not traj_image_paths:
                print(f"    demo {demo_idx}: no images found, skipping")
                continue

            processed = process_instruction(
                ambiguous_instruction,
                traj_image_paths=traj_image_paths,
                model_text=model_text,
                model_vision=model_vision,
                temperature=temperature,
            )
            if processed is None:
                print(f"    demo {demo_idx}: failed to disambiguate")
                continue

            results_list[human_idx]["disambiguated_instructions"][ambiguity_type]["per_demo"][
                str(demo_idx)
            ] = {
                "ambiguous": processed["ambiguous"],
                "disambiguated_instruction": processed["disambiguated_instruction"],
            }
            print(
                f"    demo {demo_idx}: ambiguous={processed['ambiguous']} -> "
                f"{processed['disambiguated_instruction']}"
            )

    # Save checkpoint every 12 humans
    if human_idx % 12 == 0 and human_idx > 0:
        if previous_resume_path is not None:
            os.remove(previous_resume_path)
            print(f"Removed previous resume file {previous_resume_path}")

        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        results_path = os.path.join(
            params["irl"]["data_split_config_path"],
            f"language_disambiguated_vlm_temp{temperature}_{seed}_{timestamp_str}.json",
        )
        with open(results_path, "w") as f:
            json.dump(results_list, f, indent=4, default=jsonNpEncoder)
        print(f"Saved checkpoint to {results_path}")
        previous_resume_path = results_path

# Save final results
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
results_path = os.path.join(
    params["irl"]["data_split_config_path"],
    f"language_disambiguated_vlm_temp{temperature}_{seed}_{timestamp_str}.json",
)
with open(results_path, "w") as f:
    json.dump(results_list, f, indent=4, default=jsonNpEncoder)
print(f"Saved final results to {results_path}")